In [84]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

## Clean and create Master Districts Dataset

```district_master``` - This is a cleaned up list of school districts pulled together from 2013 & 2022-2025 years of data, keeping only the districts and fields that exist in all 5 years i.e. district ID, district name, county, region, and a couple of simple flags like whether it’s a charter so I can quickly see what each district.

***To-Dos / Notes***
- This needs a lot more cleaning...
- Re-title columns
- I will eventually need to re-clean this because rn It very bare bones
- I need to consider charter is not separated
- Double-check that matching district IDs actually represent the same real district across all years
- for all incoming data I will need to remove districts not in my district_final_cut 
- and if I go back more than 2022 if one of my districts in the final cut disappears then I need to remove them from the consideration to.
- 

i.e. 
a district is not listed for any number of reasons before 2019 but in my current list of districts from 2022-2025 if i used data back to 2014 then it will skew... actually I just loaded in 2012 and added that to the set cross refrence and it narrowd down the list a little more so im just going with those ones. 




In [85]:
# Load in District data
district_2025 = pd.read_csv(r'District Infromation\2025 District Reference.csv')
district_2024 = pd.read_csv(r'District Infromation\2024 District Reference.csv')
district_2023 = pd.read_csv(r'District Infromation\2023 District Reference Information.csv')
district_2022 = pd.read_csv(r'District Infromation\2022 District Reference Information, Accountability Rating and Special Education Determination Status.csv')
district_2013 = pd.read_csv(r'District Infromation/2013 District Reference .dat', dtype=str)

In [86]:
# Promote index 0 to headers and add year columns Year to 2024 & 2025

def promote_index_to_headers(df):
    clean_headers = df.iloc[0]
    df = df[1:]
    df.columns = clean_headers
    return df


district_25 = promote_index_to_headers(district_2025)
district_25.insert(loc=0, column="Year", value="2025")

district_24 = promote_index_to_headers(district_2024)
district_24.insert(loc=0, column="Year", value="2024")

district_25.head(1)

,Year,DISTRICT,DISTNAME,COUNTY,CNTYNAME,REGION,REGNNAME,DFLCHART,DFLALTED,D_RATING,DAD_POST,ASVAB_STATUS,OUTCOME
1,2025,001902,CAYUGA ISD,001,ANDERSON,07,REGION 07: KILGORE,N,N,B,1,,Meets Requirements


In [87]:
# Strip the leading ' in front of County, Region, District (most likely there due to excel being common for analysis and it would keep those as strings not int's)
district_2022['COUNTY'] = district_2022['COUNTY'].str[1:]
district_2022['REGION'] = district_2022['REGION'].str[1:]
district_2022['DISTRICT'] = district_2022['DISTRICT'].str[1:]

district_2023['COUNTY'] = district_2023['COUNTY'].str[1:]
district_2023['REGION'] = district_2023['REGION'].str[1:]
district_2023['DISTRICT'] = district_2023['DISTRICT'].str[1:]

In [88]:
# Adding year column to 2013
district_2013.insert(loc=0, column="Year", value="2013")
district_2022.insert(loc=0, column="Year", value="2022")
district_2023.insert(loc=0, column="Year", value="2023")

district_22 = district_2022
district_23 = district_2023

In [89]:
# This is a cleaned list of school districts I’m keeping by taking only the districts that show up in every dataset (2025, 2024, 2023, 2022, and 2013), 
# based on the assumption that if a district ID exists across all years, it’s stable enough to trust and include.

# To-do: Double-check that matching district IDs actually represent the same real district across all years
d_25 = set(district_25['DISTRICT'])
d_24 = set(district_24['DISTRICT'])
d_23 = set(district_23['DISTRICT'])
d_22 = set(district_22['DISTRICT'])
d_12 = set(district_2013['DISTRICT'])

# Creating a set of districts only in all
District_final_cut = d_25 & d_24 & d_23 & d_22 & d_12
District_final_cut
len(District_final_cut)


1162

In [90]:
# Check district lengths aren't changing  
# 2025: Original List Len: 1208 New List Len: 1208
# 2024: Original List Len: 1207 New List Len: 1207
# 2023: Original List Len: 1209 New List Len: 1209
# 2022: Original List Len: 1207 New List Len: 1207

print(f'Original List Len: {len(district_22['DISTRICT'])} New Set Len: {len(d_22)}')


Original List Len: 1207 New Set Len: 1207


In [91]:
def remove_districts(districts):
    filter = districts['DISTRICT'].isin(District_final_cut)
    filtered_district = districts[filter]
    return filtered_district

In [92]:
district_25_filtered = remove_districts(district_25)
district_24_filtered = remove_districts(district_24)
district_23_filtered = remove_districts(district_23)
district_22_filtered = remove_districts(district_22)
district_13_filtered = remove_districts(district_2013)


In [93]:
dist_25_col = set(district_25_filtered.columns)
dist_24_col = set(district_24_filtered.columns)
dist_23_col = set(district_23_filtered.columns)
dist_22_col = set(district_22_filtered.columns)
dist_13_col = set(district_2013.columns)

dist_col_in_all = dist_25_col & dist_24_col & dist_23_col & dist_22_col & dist_13_col
dist_col_in_all

{'CNTYNAME',
 'COUNTY',
 'DFLALTED',
 'DFLCHART',
 'DISTNAME',
 'DISTRICT',
 'REGION',
 'Year'}

In [94]:
# now I need to make a new data frame that is a version of all of these where all the data is the same, and I want to see what wrong with the ones that arent.
# i.e. if a cdn is around an but the district name is diff or county etc

dist_df_list = [district_25_filtered, district_24_filtered, district_23_filtered, district_22_filtered, district_13_filtered]

districts_grouped = pd.concat(
    [df.loc[:, df.columns.intersection(dist_col_in_all)] for df in dist_df_list]   
)

districts_grouped

,Year,DISTRICT,DISTNAME,COUNTY,CNTYNAME,REGION,DFLCHART,DFLALTED
1,2025,001902,CAYUGA ISD,001,ANDERSON,07,N,N
2,2025,001903,ELKHART ISD,001,ANDERSON,07,N,N
3,2025,001904,FRANKSTON ISD,001,ANDERSON,07,N,N
4,2025,001906,NECHES ISD,001,ANDERSON,07,N,N
5,2025,001907,PALESTINE ISD,001,ANDERSON,07,N,N
...,...,...,...,...,...,...,...,...
1223,2013,252902,NEWCASTLE ISD,252,YOUNG,09,N,N
1224,2013,252903,OLNEY ISD,252,YOUNG,09,N,N
1225,2013,253901,ZAPATA COUNTY ISD,253,ZAPATA,01,N,N
1226,2013,254901,CRYSTAL CITY ISD,254,ZAVALA,20,N,N


In [95]:
# Check how many unique 'charter_status' values each district has
charter_check = districts_grouped.groupby('DISTRICT')['DFLALTED'].nunique()

aea_check = districts_grouped.groupby('DISTRICT')['DFLALTED'].nunique()

# IDs where the status changed (more than 1 unique value)
# inconsistent_ids = charter_check[charter_check > 1].index  # Nothing
inconsistent_ids = aea_check[aea_check > 1].index
inconsistent_ids

Index(['015807', '015808', '101804', '101837', '101842', '101864', '193801'], dtype='str', name='DISTRICT')

In [96]:
districts_grouped.shape

districts_cleaned = districts_grouped[~districts_grouped['DISTRICT'].isin(inconsistent_ids)]

print(districts_grouped.shape, districts_cleaned.shape)


(5810, 8) (5775, 8)


In [97]:
districts_cleaned

,Year,DISTRICT,DISTNAME,COUNTY,CNTYNAME,REGION,DFLCHART,DFLALTED
1,2025,001902,CAYUGA ISD,001,ANDERSON,07,N,N
2,2025,001903,ELKHART ISD,001,ANDERSON,07,N,N
3,2025,001904,FRANKSTON ISD,001,ANDERSON,07,N,N
4,2025,001906,NECHES ISD,001,ANDERSON,07,N,N
5,2025,001907,PALESTINE ISD,001,ANDERSON,07,N,N
...,...,...,...,...,...,...,...,...
1223,2013,252902,NEWCASTLE ISD,252,YOUNG,09,N,N
1224,2013,252903,OLNEY ISD,252,YOUNG,09,N,N
1225,2013,253901,ZAPATA COUNTY ISD,253,ZAPATA,01,N,N
1226,2013,254901,CRYSTAL CITY ISD,254,ZAVALA,20,N,N


In [98]:
# using 2025 as source of truth

district_clean = districts_cleaned[districts_cleaned["Year"]=='2025'].reset_index(drop=True)


district_clean.head()

,Year,DISTRICT,DISTNAME,COUNTY,CNTYNAME,REGION,DFLCHART,DFLALTED
0,2025,001902,CAYUGA ISD,001,ANDERSON,07,N,N
1,2025,001903,ELKHART ISD,001,ANDERSON,07,N,N
2,2025,001904,FRANKSTON ISD,001,ANDERSON,07,N,N
3,2025,001906,NECHES ISD,001,ANDERSON,07,N,N
4,2025,001907,PALESTINE ISD,001,ANDERSON,07,N,N


In [99]:
# renaming columns and source of truth 

district_clean.columns = ['Year', 'District ID', 'District Name', 'County ID', 'County Name', 'Region', 'Charter Flag', 'DFLALTED']
district_clean.head(2)

,Year,District ID,District Name,County ID,County Name,Region,Charter Flag,DFLALTED
0,2025,001902,CAYUGA ISD,001,ANDERSON,07,N,N
1,2025,001903,ELKHART ISD,001,ANDERSON,07,N,N


## Clean Student & Staff Datasets

```staff_master``` - This is all staff data from 2013 through 2025 merged the same way, with only the shared columns kept so everything lines up cleanly year to year.

```student_master``` - This is all student data from 2013 through 2025 merged together, keeping only the columns that exist in every year so the data is consistent across time.

***To-Dos / Notes***
- This needs a lot more cleaning...
- Account for any data masking (haven't handled that yet)
- Add testing and validation throughout the process to validate each step.
- Clean up the keys table, to account for columns that are the same thing but named differently depending on the yea, handle fields that only exist in some years but are still useful.

***Useful links***
- View had data is masked: https://rptsvr1.tea.texas.gov/perfreport/tapr/2025/masking.html (change year value 2013-2025)
- 2013 data download headers explained: https://rptsvr1.tea.texas.gov/perfreport/tapr/2013/download/taprref.html

In [100]:
# Load in all staff data
StudentStaff_Info_2013 = pd.read_csv(r"Staff & Student Data\2013 Staff & Student Information.txt", dtype=str)
StudentStaff_Info_2014 = pd.read_csv(r"Staff & Student Data\2014 Staff & Student Information.dat", dtype=str)
StudentStaff_Info_2015 = pd.read_csv(r"Staff & Student Data\2015 Staff & Student Information.dat", dtype=str)
StudentStaff_Info_2016 = pd.read_csv(r"Staff & Student Data\2016 District Staff & Student Information.dat", dtype=str)
StudentStaff_Info_2017 = pd.read_csv(r"Staff & Student Data\2017 Staff & Student Information.dat", dtype=str)
StudentStaff_Info_2018 = pd.read_csv(r"Staff & Student Data\2018 Staff, Student, and Annual Graduates.dat", dtype=str)
StudentStaff_Info_2019 = pd.read_csv(r"Staff & Student Data\2019 Staff, Student, and Annual Graduates.dat", dtype=str)
StudentStaff_Info_2020 = pd.read_csv(r"Staff & Student Data\2020 Staff, Student, and Annual Graduates.dat", dtype=str)
StudentStaff_Info_2021 = pd.read_csv(r"Staff & Student Data\2021 Staff, Student, and Annual Graduates.csv", dtype=str)
StudentStaff_Info_2022 = pd.read_csv(r"Staff & Student Data\2022 Staff, Student, and Annual Graduates.csv", dtype=str)
StudentStaff_Info_2023 = pd.read_csv(r"Staff & Student Data\2023 Staff, Student, and Annual Graduates.csv", dtype=str)
Student_Info_2024 = pd.read_csv(r"Staff & Student Data\2024 District Student Information.csv", dtype=str)
Staff_Info_2024 = pd.read_csv(r"Staff & Student Data\2024 District Staff Information.csv", dtype=str)
Student_Info_2025 = pd.read_csv(r"Staff & Student Data\2025 District Student Information.csv", dtype=str)
Staff_Info_2025 = pd.read_csv(r"Staff & Student Data\2025 District Staff Information.csv", dtype=str)


### Seprate Student and Staff Data 2013 - 2023

- When I downloaded the datasets for student and staff information 2013 and 2023 the student staff data was combined. I will want to separate these later but to make managing this many tables scalable I will throw then in a dict. ```Note: I will add 2024 and 2025 separated student and staff DataFrames to a dictionary later.```
- Load in the student staff key files

Future Goal: Combining all student and staff data downloads in master keys based on all unique rows but i think a function will be quicker for me to do now so I'll merge the headers later

In [101]:
# Add student and staff 2013 and 2023 combined dataframes to a dictionary.

student_staff_dfs = {2013:StudentStaff_Info_2013, 
                      2014:StudentStaff_Info_2014, 
                      2015:StudentStaff_Info_2015, 
                      2016:StudentStaff_Info_2016, 
                      2017:StudentStaff_Info_2017,
                      2018:StudentStaff_Info_2018,
                      2019:StudentStaff_Info_2019, 
                      2020:StudentStaff_Info_2020,
                      2021:StudentStaff_Info_2021,
                      2022:StudentStaff_Info_2022,
                      2023:StudentStaff_Info_2023}
type(student_staff_dfs)

dict

In [102]:
# Loading in all the student and staff keys for 2013 to 2024

student_key_tables ={}
staff_key_tables = {}

for year in range(2013, 2024):
    # Load in files with a year variable
    student_key = pd.read_csv(rf"Staff & Student Keys\{year}_TAPR_Advanced_Data_Download_Student.csv")
    staff_key = pd.read_csv(rf"Staff & Student Keys\{year}_TAPR_Advanced_Data_Download_Staff.csv")

    student_key_tables[year] = student_key
    staff_key_tables[year] = staff_key

print(len(student_key_tables), len(staff_key_tables))


11 11


In [103]:
# For the 2024 and 2025 school years, data sets had the header a human readable name I stripped that for now and promoted index 0 which used the old id naming conversion

# Note: I'll eventually need to save those header
def PromoteFirstRow(df, year):  
    new_headers = df.iloc[0]
    new_df = df.iloc[1:]
    new_df.columns = new_headers
    new_df = new_df.reset_index(drop=True)
    key_df = pd.DataFrame(new_headers)
    key_df = key_df.reset_index()
    key_df.columns = ["LABEL", "NAME"]

    return new_df, key_df

Student_Info_2024, key_df_stu_24 = PromoteFirstRow(Student_Info_2024, 2024)
student_key_tables[2024] = key_df_stu_24

Student_Info_2025, key_df_stu_25= PromoteFirstRow(Student_Info_2025, 2025)
student_key_tables[2025] = key_df_stu_25

Staff_Info_2024, key_df_staff_24= PromoteFirstRow(Staff_Info_2024, 2024)
staff_key_tables[2024] = key_df_staff_24

Staff_Info_2025, key_df_staff_25= PromoteFirstRow(Staff_Info_2025, 2025)
staff_key_tables[2025] = key_df_staff_25


In [104]:
# CREATE: two dictionary one holding all the student data another holding all the staff data 

student_tables = {}
staff_tables = {}

for year in range(2013, 2024):
    # In the dictionary student_tables we have {year: DataFrame} i want to copy the data from the respective 


    student_tables[year] = student_staff_dfs[year][student_key_tables[year]['NAME']].copy()
    staff_tables[year] = student_staff_dfs[year][staff_key_tables[year]['NAME']].copy()

    student_tables[year].insert(loc=0, column="Year", value=year)
    staff_tables[year].insert(loc=0, column="Year", value=year)

    if year == 2023:

        
        student_tables[2024] = Student_Info_2024
        student_tables[2025] = Student_Info_2025
        student_tables[2024].insert(loc=0, column="Year", value=2024)
        student_tables[2025].insert(loc=0, column="Year", value=2025)

        staff_tables[2024] = Staff_Info_2024
        staff_tables[2025] = Staff_Info_2025
        staff_tables[2024].insert(loc=0, column="Year", value=2024)
        staff_tables[2025].insert(loc=0, column="Year", value=2025)

# Removing the ' character from districts column, likely put there for when excel was used to know the field was a string
staff_tables[2021]['DISTRICT'] = staff_tables[2021]['DISTRICT'] .str[1:]
staff_tables[2022]['DISTRICT'] = staff_tables[2022]['DISTRICT'] .str[1:]
staff_tables[2023]['DISTRICT'] = staff_tables[2023]['DISTRICT'] .str[1:]

student_tables[2021]['DISTRICT'] = student_tables[2021]['DISTRICT'] .str[1:]
student_tables[2022]['DISTRICT'] = student_tables[2022]['DISTRICT'] .str[1:]
student_tables[2023]['DISTRICT'] = student_tables[2023]['DISTRICT'] .str[1:]

In [105]:
# doing a basic test to see if the shapes make sense not sure why its plus one but not worries enough to check rn... i feel like it something to do with counting a header or index
for i in range(2013, 2024):    
    print(f"[{i}]: New Shape: {staff_tables[i].shape}, Expected shape: ({student_staff_dfs[i].shape[0]}, {staff_key_tables[i].shape[0]})")

[2013]: New Shape: (1228, 138), Expected shape: (1228, 137)
[2014]: New Shape: (1227, 138), Expected shape: (1227, 137)
[2015]: New Shape: (1219, 138), Expected shape: (1219, 137)
[2016]: New Shape: (1207, 138), Expected shape: (1207, 137)
[2017]: New Shape: (1203, 151), Expected shape: (1203, 150)
[2018]: New Shape: (1200, 155), Expected shape: (1200, 154)
[2019]: New Shape: (1201, 155), Expected shape: (1201, 154)
[2020]: New Shape: (1202, 116), Expected shape: (1202, 115)
[2021]: New Shape: (1204, 121), Expected shape: (1204, 120)
[2022]: New Shape: (1207, 127), Expected shape: (1207, 126)
[2023]: New Shape: (1209, 121), Expected shape: (1209, 120)


#### Current Data State (Checkpoint)

```student_tables = {}```

```staff_tables = {}```

```student_key_tables ={}```

```staff_key_tables = {}```

Note: *If comparing the data shown from this year’s report to, previous reports, use the data displayed under Membership.* [Source](https://tea.texas.gov/texas-schools/accountability/academic-accountability/performance-reporting/2020-profile-glossary.pdf)


Having difficulty understanding why the student count appears in 2023 data

In [106]:
# FILTER: Remove rows from LABEL column containing ['Percent', 'Rate', 'Ratio', '%', 'Enrollment', "Average"]

"""
staff_key_tables: Dict[df_1, df_2, ...]

"""

bad_strings = ['Percent', 'Rate', 'Ratio', '%', 'Enrollment', "Average", "Mobility", 'DAEP']
pattern = "|".join(bad_strings)

for year in range(2013, 2026):
    student_key_tables[year] = student_key_tables[year][~student_key_tables[year]["LABEL"].str.contains(pattern, na=False)]
    staff_key_tables[year] = staff_key_tables[year][~staff_key_tables[year]["LABEL"].str.contains(pattern, na=False)]

    student_key_tables[year] = student_key_tables[year].reset_index(drop=True)
    staff_key_tables[year] = staff_key_tables[year].reset_index(drop=True)

# staff_key_tables[2019]

In [107]:
# FILTER: staff & student key tables to only rows whose NAME matches a pattern, producing clean indexed subsets.

"""
Data cleaning feel like sculpting clay
your change the shape, fold it, remove pars
"""
student_string = r"DPE"
staff_string = r"" 

student_24 = student_key_tables[2024]
keep_columns_student_df = student_24[student_24['NAME'].apply(str).str.contains(student_string)].copy()
keep_columns_student_df = keep_columns_student_df.reset_index(drop=True)

staff_24 = staff_key_tables[2024]
keep_columns_staff_df = staff_24[staff_24['NAME'].apply(str).str.contains(staff_string)].copy()
keep_columns_staff_df = keep_columns_staff_df.reset_index(drop=True)

print(keep_columns_student_df.shape, student_key_tables[2024].shape)

(51, 2) (66, 2)


In [108]:
# VISUAL: Summaries what was retained/removed after the filtering step above. - staff & student key tables

print("=" * 60)
print("Checkpoint: Staff & Student key tables ")
print("=" * 60)

# ── Student ────────────────────────────────────────────────────────────────
student_original = student_key_tables[2024].shape[0]
student_kept     = keep_columns_student_df.shape[0]
student_removed  = student_original - student_kept

print(f"\n  ── Student Columns DataFrame ──")
print(f"    Filter string   : '{student_string}'")
print(f"    Original rows   : {student_original}")
print(f"    Rows kept       : {student_kept}")
print(f"    Rows removed    : {student_removed}")

# ── Staff ──────────────────────────────────────────────────────────────────
staff_original   = staff_key_tables[2024].shape[0]
staff_kept       = keep_columns_staff_df.shape[0]
staff_removed    = staff_original - staff_kept

print(f"\n  ── Staff Columns DataFrame ──")
print(f"    Filter string   : '{staff_string}'")
print(f"    Original rows   : {staff_original}")
print(f"    Rows kept       : {staff_kept}")
print(f"    Rows removed    : {staff_removed}")

Checkpoint: Staff & Student key tables 

  ── Student Columns DataFrame ──
    Filter string   : 'DPE'
    Original rows   : 66
    Rows kept       : 51
    Rows removed    : 15

  ── Staff Columns DataFrame ──
    Filter string   : ''
    Original rows   : 63
    Rows kept       : 63
    Rows removed    : 0


In [109]:
# CLEAN VALUES: Remove every thing before the colon in the keys table: aka the "district 2024"

keep_columns_student_df['LABEL'] = keep_columns_student_df['LABEL'].str.split(':').str[-1].str.strip()
keep_columns_staff_df['LABEL'] = keep_columns_staff_df['LABEL'].str.split(':').str[-1].str.strip()


In [110]:
# function to filter columns

id_columns = ['DISTRICT', 'Year']

def filter_columns(keys_df, dirty_df):
    key_cols = list(keys_df["NAME"])
    allowed_cols = key_cols + id_columns
    filtered_df = dirty_df.reindex(columns=allowed_cols)
    
    return filtered_df

In [111]:
# Creating long df for student and staff combined 2013-2025 data sets. 

student_long_df = []
staff_long_df = []

for year in range(2013, 2026):
    # keep columns in that I cleaned
    filtered_student_df = filter_columns(keep_columns_student_df, student_tables[year])
    filtered_staff_df = filter_columns(keep_columns_staff_df, staff_tables[year])

    student_long_df.append(filtered_student_df)
    staff_long_df.append(filtered_staff_df)

student_long_df = pd.concat(student_long_df, ignore_index=True)
staff_long_df   = pd.concat(staff_long_df, ignore_index=True)

column_to_move = student_long_df.pop('Year')
student_long_df.insert(0, 'Year', column_to_move)

column_to_move = staff_long_df.pop('Year')
staff_long_df.insert(0, 'Year', column_to_move)



In [112]:
staff_long_df

,Year,DISTRICT,DISTNAME,DPSTTOFC,DPSUTOFC,DPSSTOFC,DPSCTOFC,DPSETOFC,DPSTTOST,DPSUTOST,DPSSTOST,DPST00FC,DPST01FC,DPST06FC,DPST11FC,DPST21FC,DPST30FC,DPST00ST,DPST01ST,DPST06ST,DPST11ST,DPST21ST,DPST30ST,DPSTNOFC,DPSTBAFC,DPSTMSFC,DPSTPHFC,DPSECOFC,DPSPTOFC,DPSRTOPC,DPSRTOFC,DPSBTOPC,DPSBTOFC,DPSXTOFC,DPSATOFC,DPSOTOFC,DPSTINFC,DPSTPIFC,DPSTASFC,DPSTBLFC,DPSTHIFC,DPSTWHFC,DPSTTWFC,DPSTMAFC,DPSTFEFC,DPSTREFC,DPSTVOFC,DPSTBIFC,DPSTCOFC,DPSTGIFC,DPSTSPFC,DPSTOPFC,DPSTURNN,DPSTURND,DPSHEXPT,DPSHTENT,DPSLEXPT,DPSLTENT,DPSAMIFC,DPSTEXPT,DPSTTENT,DMSTHCNT,DEXPHCNT,DRECHCNT,DISTRICT
0,2013,001902,NaN,54.2,7,3,1.6,11.6,2444371.4,371739.2,197454,1,6,11.8,20,NaN,NaN,30872,232733,475936,929229.6,NaN,NaN,0,47.4,6.8,0,22.8,65.8,NaN,NaN,NaN,NaN,28.8,106,2,0,0,0,2,3,49.2,0,12.8,41.4,45.4,3.2,0.4,0,0,5.4,0,5.4,54.2,NaN,NaN,NaN,NaN,14,NaN,NaN,NaN,NaN,NaN,001902
1,2013,001903,NaN,96.6,7.6,5,3,15.6,3915026.2,347667.8,300835,14.2,9,21,33,NaN,NaN,514659.4,274061.2,752868.2,1439600,NaN,NaN,2,81.2,13.4,0,0,112.2,NaN,NaN,NaN,NaN,28,155.8,0,0,0,0,1,3,91.6,1,15,81.6,77.2,5.4,0,0.6,2,11.6,0,19,96.4,NaN,NaN,NaN,NaN,7,NaN,NaN,NaN,NaN,NaN,001903
2,2013,001904,NaN,55.8,5.2,3,2,10,2370621.4,258715.4,209220,2,11,7,22.8,NaN,NaN,59451,352978.4,262398.8,1058773.4,NaN,NaN,0,49.8,6,0,0,66,NaN,NaN,NaN,NaN,28.6,104.6,1,1,0,0,1,1,52.8,0,13,42.8,42.4,8,0,0.6,0,4,0.8,20.4,65.8,NaN,NaN,NaN,NaN,8.4,NaN,NaN,NaN,NaN,NaN,001904
3,2013,001906,NaN,34,2,2,1,6,1417515,88239,130000,2,8,5,13,NaN,NaN,58793,256896.8,196655.6,586946.6,NaN,NaN,0,33,1,0,0,39,NaN,NaN,NaN,NaN,14.8,59.8,0,0,0,0,1,0,33,0,9,25,27.2,2.8,0,0.4,0.6,3,0,4,33,NaN,NaN,NaN,NaN,4.8,NaN,NaN,NaN,NaN,NaN,001906
4,2013,001907,NaN,252.8,27.8,16.4,8,70,10729855.2,1352453.8,1103779,24.8,77,50,50.2,NaN,NaN,798104.4,2746109.6,2073067.6,2394626,NaN,NaN,3,206.4,41.4,2,0,304.8,NaN,NaN,NaN,NaN,89,464,0,0,0,1,21,17,208.8,5,60.8,192,172.6,13.8,4.6,26,0.6,22.6,12.4,59.4,234.6,NaN,NaN,NaN,NaN,109.2,NaN,NaN,NaN,NaN,NaN,001907
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15717,2025,252902,NEWCASTLE ISD,24.8,0.7,2,1,10.2,1479470,49629,165535,1.8,2.8,6.6,4.8,7.9,0.9,74320,153242.9,343537.9,319969.4,560911,27488.7,1.8,15.5,7.5,0,0,28.5,0,0,0,0,5.9,44.6,0,0,0,0,0,2,22.8,0,5,19.8,18.4,2.6,0,1.2,0,2.6,0,1.8,22.5,7,3,0,0,4.8,373.4,170.2,NaN,NaN,NaN,252902
15718,2025,252903,OLNEY ISD,73.3,7.7,6.3,1,1,4098574.3,515342.9,499474.8,3.9,22.1,13.5,17.7,9.4,6.7,152786.2,963453.4,712909,1086081.1,629750,553594.5,3.9,51.1,18.3,0,0,88.3,1,3,0,1,33,122.3,0,0,0,1,2,5.8,64.6,0,14.4,58.9,52.7,7.8,0.5,0.2,0,8.5,3.5,17.1,72.4,21,12,2,1,23,1175.8,502.7,NaN,1,NaN,252903
15719,2025,253901,ZAPATA COUNTY ISD,240.2,47.4,14.3,11,62.7,13664248.8,3345159.2,1183970.5,36.9,58,26,75.2,37.1,7,1764700.5,2961178.4,1435489.6,4565026.7,2459849,478004.5,2,188.9,48.3,1,0,312.9,0,14,0,5,174.5,550.1,0,0,0,0,0,233.2,6,1,44,196.2,148.6,13.8,33.6,0,0,33.9,10.2,17.9,235.9,33,33,52,52,540.1,2718.4,2449,NaN,NaN,NaN,253901
15720,2025,254901,CRYSTAL CITY ISD,107.8,22.9,13.2,7,62.8,6129788.5,1547240.3,980979.2,14,29,16.8,29,16,3,688676.8,1469279.5,935589.5,1748961.5,1074193.2,213088.1,0,95,12.8,0,0,150.9,0,8,0,1,112.7,326.4,0,0,0,0,1,99.8,7,0,35,72.8,86.4,7.1,2.8,0.2,0.7,8,2.6,17,112.4,16,16,18,12,317.4,1182.3,1058.4,NaN,NaN,NaN,254901


In [113]:
# FILTER: Removing duplicate "DISTRICTS" column at the end of staff_long_df
columns_count_before = staff_long_df.shape[1]

if staff_long_df.columns.duplicated().any():
    staff_long_df = staff_long_df.iloc[:, :-1] 
    print(f"Column Removed")
else: print("No Columns Removed")

columns_count_after = staff_long_df.shape[1]

print(f'Before: {columns_count_before} After: {columns_count_after}')

Column Removed
Before: 65 After: 64


In [114]:
staff_long_df

,Year,DISTRICT,DISTNAME,DPSTTOFC,DPSUTOFC,DPSSTOFC,DPSCTOFC,DPSETOFC,DPSTTOST,DPSUTOST,DPSSTOST,DPST00FC,DPST01FC,DPST06FC,DPST11FC,DPST21FC,DPST30FC,DPST00ST,DPST01ST,DPST06ST,DPST11ST,DPST21ST,DPST30ST,DPSTNOFC,DPSTBAFC,DPSTMSFC,DPSTPHFC,DPSECOFC,DPSPTOFC,DPSRTOPC,DPSRTOFC,DPSBTOPC,DPSBTOFC,DPSXTOFC,DPSATOFC,DPSOTOFC,DPSTINFC,DPSTPIFC,DPSTASFC,DPSTBLFC,DPSTHIFC,DPSTWHFC,DPSTTWFC,DPSTMAFC,DPSTFEFC,DPSTREFC,DPSTVOFC,DPSTBIFC,DPSTCOFC,DPSTGIFC,DPSTSPFC,DPSTOPFC,DPSTURNN,DPSTURND,DPSHEXPT,DPSHTENT,DPSLEXPT,DPSLTENT,DPSAMIFC,DPSTEXPT,DPSTTENT,DMSTHCNT,DEXPHCNT,DRECHCNT
0,2013,001902,NaN,54.2,7,3,1.6,11.6,2444371.4,371739.2,197454,1,6,11.8,20,NaN,NaN,30872,232733,475936,929229.6,NaN,NaN,0,47.4,6.8,0,22.8,65.8,NaN,NaN,NaN,NaN,28.8,106,2,0,0,0,2,3,49.2,0,12.8,41.4,45.4,3.2,0.4,0,0,5.4,0,5.4,54.2,NaN,NaN,NaN,NaN,14,NaN,NaN,NaN,NaN,NaN
1,2013,001903,NaN,96.6,7.6,5,3,15.6,3915026.2,347667.8,300835,14.2,9,21,33,NaN,NaN,514659.4,274061.2,752868.2,1439600,NaN,NaN,2,81.2,13.4,0,0,112.2,NaN,NaN,NaN,NaN,28,155.8,0,0,0,0,1,3,91.6,1,15,81.6,77.2,5.4,0,0.6,2,11.6,0,19,96.4,NaN,NaN,NaN,NaN,7,NaN,NaN,NaN,NaN,NaN
2,2013,001904,NaN,55.8,5.2,3,2,10,2370621.4,258715.4,209220,2,11,7,22.8,NaN,NaN,59451,352978.4,262398.8,1058773.4,NaN,NaN,0,49.8,6,0,0,66,NaN,NaN,NaN,NaN,28.6,104.6,1,1,0,0,1,1,52.8,0,13,42.8,42.4,8,0,0.6,0,4,0.8,20.4,65.8,NaN,NaN,NaN,NaN,8.4,NaN,NaN,NaN,NaN,NaN
3,2013,001906,NaN,34,2,2,1,6,1417515,88239,130000,2,8,5,13,NaN,NaN,58793,256896.8,196655.6,586946.6,NaN,NaN,0,33,1,0,0,39,NaN,NaN,NaN,NaN,14.8,59.8,0,0,0,0,1,0,33,0,9,25,27.2,2.8,0,0.4,0.6,3,0,4,33,NaN,NaN,NaN,NaN,4.8,NaN,NaN,NaN,NaN,NaN
4,2013,001907,NaN,252.8,27.8,16.4,8,70,10729855.2,1352453.8,1103779,24.8,77,50,50.2,NaN,NaN,798104.4,2746109.6,2073067.6,2394626,NaN,NaN,3,206.4,41.4,2,0,304.8,NaN,NaN,NaN,NaN,89,464,0,0,0,1,21,17,208.8,5,60.8,192,172.6,13.8,4.6,26,0.6,22.6,12.4,59.4,234.6,NaN,NaN,NaN,NaN,109.2,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15717,2025,252902,NEWCASTLE ISD,24.8,0.7,2,1,10.2,1479470,49629,165535,1.8,2.8,6.6,4.8,7.9,0.9,74320,153242.9,343537.9,319969.4,560911,27488.7,1.8,15.5,7.5,0,0,28.5,0,0,0,0,5.9,44.6,0,0,0,0,0,2,22.8,0,5,19.8,18.4,2.6,0,1.2,0,2.6,0,1.8,22.5,7,3,0,0,4.8,373.4,170.2,NaN,NaN,NaN
15718,2025,252903,OLNEY ISD,73.3,7.7,6.3,1,1,4098574.3,515342.9,499474.8,3.9,22.1,13.5,17.7,9.4,6.7,152786.2,963453.4,712909,1086081.1,629750,553594.5,3.9,51.1,18.3,0,0,88.3,1,3,0,1,33,122.3,0,0,0,1,2,5.8,64.6,0,14.4,58.9,52.7,7.8,0.5,0.2,0,8.5,3.5,17.1,72.4,21,12,2,1,23,1175.8,502.7,NaN,1,NaN
15719,2025,253901,ZAPATA COUNTY ISD,240.2,47.4,14.3,11,62.7,13664248.8,3345159.2,1183970.5,36.9,58,26,75.2,37.1,7,1764700.5,2961178.4,1435489.6,4565026.7,2459849,478004.5,2,188.9,48.3,1,0,312.9,0,14,0,5,174.5,550.1,0,0,0,0,0,233.2,6,1,44,196.2,148.6,13.8,33.6,0,0,33.9,10.2,17.9,235.9,33,33,52,52,540.1,2718.4,2449,NaN,NaN,NaN
15720,2025,254901,CRYSTAL CITY ISD,107.8,22.9,13.2,7,62.8,6129788.5,1547240.3,980979.2,14,29,16.8,29,16,3,688676.8,1469279.5,935589.5,1748961.5,1074193.2,213088.1,0,95,12.8,0,0,150.9,0,8,0,1,112.7,326.4,0,0,0,0,1,99.8,7,0,35,72.8,86.4,7.1,2.8,0.2,0.7,8,2.6,17,112.4,16,16,18,12,317.4,1182.3,1058.4,NaN,NaN,NaN


In [115]:
# FILTER: Only keep district in my df - district_clean 

'''
Split the data: Call the .groupby() method on your DataFrame, passing the column name (as a string) 
or multiple column names (as a list of strings) you want to group by. This creates a GroupBy object.

grouped = df.groupby('Category')
# or
grouped_multiple = df.groupby(['Category', 'Subcategory'])
'''

allowed_district = list(district_clean['District ID'])

def filter_districts(df):


    df = df[df['DISTRICT'].isin(allowed_district)]
    return df.reset_index(drop=True)
    df.groupby('DISTRICT')

    student_df.groupby('DISTRICT').filter(lambda x: (x[''] >= 30).all())

# before = student_long_df.shape[0]
#student_long_df = filter_districts(student_long_df)
#staff_long_df = filter_districts(staff_long_df)

#print(f"before: {before} After: {student_long_df.shape[0]}")

# staff_long_df.head()

type(allowed_district[0])
#student_long_df[student_long_df["DISTRICT"]].isin(allowed_district)
#student_long_df

len(student_long_df.groupby(["DISTRICT"]))


1282

In [116]:
# Replace ID headers with human readable names
def replace_col_headers(keys_df, long_df):
    label_to_name = (keys_df.set_index("NAME")["LABEL"].to_dict())
    long_df.rename(columns=label_to_name, inplace=True)
    return long_df

student_long_df = replace_col_headers(keep_columns_student_df, student_long_df)
staff_long_df = replace_col_headers(keep_columns_staff_df, staff_long_df)

In [117]:
# manual header cleaning
district_col_move = student_long_df.pop('DISTRICT')
student_long_df.insert(1, "DISTRICT", district_col_move)
student_long_df.head()

staff_long_df.rename(columns={'6 Digit County District Number': 'DISTRICT'}, inplace=True)

In [118]:
# Create compost ID. Year + District string
student_long_df.insert(loc=0, column="YearDistrict ID", value=(student_long_df['Year'].astype(str) + student_long_df["DISTRICT"].astype(str)))
staff_long_df.insert(loc=0, column="YearDistrict ID", value = (staff_long_df['Year'].astype(str)  + staff_long_df["DISTRICT"].astype(str))) 

# student_long_df

In [119]:
# Handle masking
student_long_df = student_long_df.replace(["n/a", '-1', '-2', '-3', "•"], np.nan)
staff_long_df = staff_long_df.replace(["n/a", '-1', '-2', '-3', "•"], np.nan)

In [120]:
# filter down dataframes to 2017 - 2025
student_analysis_df = student_long_df[student_long_df['Year'] >= 2017]
staff_analysis_df = staff_long_df[staff_long_df['Year'] >= 2017]

In [121]:
# Add cleaned District descriptive attributes

district_dimensions = district_clean[["District ID", 'District Name', 'County ID', 'County Name', 'Region', 'Charter Flag', 'DFLALTED']]


def add_district_dim(df):
    df = df.merge(
        district_dimensions,
        left_on="DISTRICT",
        right_on="District ID",
        how="left"
        
    )
    df = df.drop(columns=["District ID"])

    return df

student_analysis_df = add_district_dim(student_analysis_df)
staff_analysis_df = add_district_dim(staff_analysis_df)

In [122]:
# Converting the numeric values back to numbers from strings
string_cols = [
    "Year",
    "DISTRICT",
    "District Name",
    "County ID",
    "County Name",
    "Region",
    "Charter Flag",
    "DFLALTED",
]

def coerce_numeric_except(df, string_cols):
    df = df.copy()

    numeric_cols = df.columns.difference(string_cols)

    df[numeric_cols] = df[numeric_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    return df

student_analysis_df = coerce_numeric_except(
    student_analysis_df,
    string_cols
)

staff_analysis_df = coerce_numeric_except(
    staff_analysis_df,
    string_cols
)

In [123]:
# Export files to Clean Datasets
student_analysis_df.to_csv(r'Clean Datasets/student_analysis_df.csv', index=False)
staff_analysis_df.to_csv(r'Clean Datasets/staff_analysis_df.csv', index=False)


In [124]:
x = student_analysis_df.dtypes
x

YearDistrict ID                                int64
Year                                           int64
DISTRICT                                         str
EE Count                                       int64
PK Count                                       int64
KG Count                                       int64
01 Count                                       int64
02 Count                                       int64
03 Count                                       int64
04 Count                                       int64
05 Count                                       int64
06 Count                                       int64
07 Count                                       int64
08 Count                                       int64
09 Count                                       int64
10 Count                                       int64
11 Count                                       int64
12 Count                                       int64
All Students Count                            